# 09 · CADD — one score for every kind of variant, fetched live

**CADD** (Combined Annotation Dependent Depletion) collapses dozens of genomic
annotations into a single deleteriousness score. It is the only tool in this series with
no local file at all: there is nothing to download or build, just a per-variant call to a
live API.

---

### Why a general deleteriousness score matters for CFTR

Every other predictor here answers a question about *one class of variant*. AlphaMissense,
EVE, ESM1b, REVEL and PrimateAI score amino-acid substitutions and are blind to everything
else (tools/02–06). SpliceAI and Pangolin score splice disruption and say nothing about
protein folding (tools/07–08). But CFTR disease alleles do not respect those boundaries —
the same gene carries missense (`G551D`), nonsense, frameshifts (`F508del` is an in-frame
3-bp deletion), canonical splice-site changes, deep-intronic cryptic-exon creators, and
whole-exon deletions.

So for most of a real CFTR variant list, **most of the tools have nothing to say**. CADD
is the one score that returns a number for a missense change, a splice variant and a deep
intronic position on the *same scale*, which makes it the natural first pass over a mixed
list before you reach for the specialist that actually fits each variant.

That breadth is also its limitation, and the trade is worth being explicit about: one
number carries no mechanism. A high CADD says "this position looks unusually constrained
given everything we know about it" — not "this breaks splicing" or "this misfolds the
protein". It ranks; it does not explain. Use it to *order* a list, then use the tool that
models the actual mechanism to understand the top of it.

### What the score means

CADD is **PHRED-scaled** against all ~8.6 billion possible substitutions in the genome:

- **PHRED ≥ 15** → roughly the **top 3%** most deleterious
- **PHRED ≥ 20** → roughly the **top 1%**

A raw score also comes back; the PHRED value is the one worth quoting, because it is a
rank rather than an absolute.

> ✅ **REAL / LIVE.** `fetch_cadd(...)` (defined below) calls the live CADD v1.7 API, so
> this notebook needs **network access** and nothing else — no download, no build cell, no
> `data/` file. CADD PHRED **≥ 15 ~ top 3%**, **≥ 20 ~ top 1%**.
>
> **Version, and why it is the anchor.** The version is pinned in the URL itself
> (`GRCh38-v1.7`) and returned with every score below. That is the closest thing a live
> API has to a release stamp, and it matters for the same reason it does elsewhere in this
> series: **CADD v1.7 was released 2024-01-05**, so it bounds what could have informed the
> model. Section 3 works through why that date is later — and more awkward — than it looks.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # repo root, for toolkit.py
import toolkit as tk
import pandas as pd, numpy as np
%matplotlib inline

## 1 · Calling the API

`fetch_cadd(chrom, pos, ref, alt)` queries the **live CADD v1.7 REST API** and returns
`{'cadd_raw', 'cadd_phred', 'cadd_version'}` — or `None` scores on a miss. The endpoint is

```
https://cadd.gs.washington.edu/api/v1.0/GRCh38-v1.7/{chr}:{pos}-{pos}
```

which returns every alternate allele at that position; the helper picks the row matching
the `ref`/`alt` you asked for.

> **Strand: no gotcha for CFTR.** CFTR sits on the genomic **plus (forward) strand**
> (7q31.2), so a change written on the *coding* strand (say `C>T`) appears on the genome
> as the **same** alleles — you do **not** complement it. CADD is indexed on the plus
> strand, so coding-strand alleles match directly.
>
> The gotcha that *does* bite is the **genome build**: GRCh37 and GRCh38 CFTR coordinates
> differ by ~200 kb, so a build mix-up matches nothing and returns an empty result rather
> than an error. The endpoint pins GRCh38 explicitly for that reason.

> **Reproducibility.** A live API is the least reproducible source in this toolkit: there
> is no file to hash and no snapshot to keep. Pin the version in the URL (done), record it
> alongside every score (done), and **cache responses** if you need a run to be repeatable
> or to work offline — a version bump would change every number.

In [2]:
import time, requests

CADD_VERSION = "GRCh38-v1.7"      # pinned in the URL; returned with every score
CADD_API = "https://cadd.gs.washington.edu/api/v1.0/{ver}/{chrom}:{pos}-{pos}"


def fetch_cadd(chrom: str, pos: int, ref: str, alt: str, delay_sec: float = 0.3) -> dict:
    """Score ONE variant with the live CADD API. No local file involved.

    Returns dict(cadd_raw, cadd_phred, cadd_version); the scores are None on a miss.
    The version travels with the score deliberately -- a PHRED value is only meaningful
    against the model that produced it, and this is a live endpoint that will move on.

    NOTE ON STRAND: CFTR is on the genomic PLUS strand (7q31.2), so a coding change
    (e.g. C>T) is reported on the genome as the SAME alleles -- no complementing needed.
    This helper also tries the complement, purely as a guard against upstream tables that
    (wrongly, for CFTR) report minus-strand alleles; for correct input it never fires.
    """
    url = CADD_API.format(ver=CADD_VERSION, chrom=chrom, pos=pos)
    comp = {"A": "T", "T": "A", "C": "G", "G": "C"}
    out = {"cadd_raw": None, "cadd_phred": None, "cadd_version": CADD_VERSION}
    try:
        resp = requests.get(url, timeout=15)
        resp.raise_for_status()          # a 4xx/5xx must not look like "no score"
        data = resp.json()
    except Exception as exc:
        return {**out, "error": f"{type(exc).__name__}: {exc}"}
    for rec in data[1:] if data else []:
        if len(rec) < 6:
            continue
        r, a = rec[2], rec[3]
        if (r == ref and a == alt) or (r == comp.get(ref) and a == comp.get(alt)):
            time.sleep(delay_sec)        # be polite to a public endpoint
            return {**out, "cadd_raw": float(rec[4]), "cadd_phred": float(rec[5])}
    time.sleep(delay_sec)
    return out

In [3]:
# LIVE call 1 — an intronic position near the benign end of the scale.
res1 = fetch_cadd('7', 117548628, 'G', 'A')
print('7:117548628 G>A  ->', res1)
print(f"  CADD PHRED = {res1['cadd_phred']}  (low: nothing here looks constrained)")

7:117548628 G>A  -> {'cadd_raw': -0.964845, 'cadd_phred': 0.028, 'cadd_version': 'GRCh38-v1.7'}
  CADD PHRED = 0.028  (low: nothing here looks constrained)


A real, low score — the API answered, and the answer is "unremarkable". Now a variant that
should score high: `c.2657+5G>A`, a donor-region change, queried with the plus-strand
alleles `G`/`A` taken from CFTR2's authoritative coordinates.

In [4]:
# LIVE call 2 — a known CF-causing splice variant at its authoritative coordinate.
res2 = fetch_cadd('7', 117602868, 'G', 'A')   # c.2657+5G>A
print('7:117602868 G>A  ->', res2)
phred2 = res2['cadd_phred']
if phred2 is not None:
    band = 'top ~1% (>=20)' if phred2 >= 20 else ('top ~3% (>=15)' if phred2 >= 15 else 'below 15')
    print(f'  CADD PHRED = {phred2}  ->  {band}')

7:117602868 G>A  -> {'cadd_raw': 3.907754, 'cadd_phred': 23.8, 'cadd_version': 'GRCh38-v1.7'}
  CADD PHRED = 23.8  ->  top ~1% (>=20)


## 2 · The splice panel, scored by CADD

The same fixed panel of famous CFTR **splice** variants runs through tools/07–09, so one
set of variants can be followed across the series. The variant list is `tk.A2_KNOWN_CDNA`
and the legacy names are `tk.A2_KNOWN_LEGACY`, both in `toolkit.py`.

CFTR2's coordinates are joined on but not printed — its terms forbid republishing any
portion of its content. Comparing these numbers against SpliceAI's and Pangolin's is a
benchmark rather than a tool walkthrough, so it belongs in the `predict/` pipeline.

In [5]:
# One live API call per variant, joined on CFTR2's authoritative GRCh38 coordinates.
cf = tk.load_cftr2()
targets = cf[cf['cdna_name'].isin(tk.A2_KNOWN_CDNA)].dropna(subset=['grch38_pos'])
rows = []
for _, v in targets.iterrows():
    res = fetch_cadd('7', int(v['grch38_pos']), v['grch38_ref'], v['grch38_alt'])
    rows.append({'cdna_name': v['cdna_name'],
                 'legacy_name': tk.A2_KNOWN_LEGACY.get(v['cdna_name'], ''),
                 'cadd_phred': res['cadd_phred'],
                 'cadd_version': res['cadd_version']})
panel = pd.DataFrame(rows)
print(f"{int(panel['cadd_phred'].notna().sum())} scored / {len(panel)} looked up "
      f"/ {len(tk.A2_KNOWN_CDNA)} in the panel")
panel

5 scored / 5 looked up / 5 in the panel


,cdna_name,legacy_name,cadd_phred,cadd_version
0,c.3718-2477C>T,3849+10kbC>T,10.62,GRCh38-v1.7
1,c.2657+5G>A,2789+5G>A,23.80,GRCh38-v1.7
2,c.3140-26A>G,3272-26A>G,25.00,GRCh38-v1.7
3,c.2988+1G>A,3120+1G>A,33.00,GRCh38-v1.7
4,c.1680-886A>G,1811+1634A>G,34.00,GRCh38-v1.7


## 3 · What CADD is made of, and why that complicates the date

CADD is **not** trained on clinical labels. Its two training sets are *proxy* classes: per
the v1.7 release notes, "more than 14 million human lineage derived sequence alterations"
as proxy-neutral, against "a size matched set of simulated variants" as proxy-deleterious,
with the model trained "to contrast the proxy-neutral and proxy-deleterious sets based on
available genomic annotations". No ClinVar, no HGMD. So its **direct** circularity against
a clinical truth set is low — the concern that dominates REVEL
([`05_revel.ipynb`](05_revel.ipynb) section 2) does not apply the same way here.

**But CADD is a meta-model, and this repo is inside it.** The v1.7 feature table lists,
among ~120 annotations:

| CADD v1.7 feature | also a tool in this series |
|---|---|
| `SpliceAI-acc-gain`, `SpliceAI-acc-loss`, `SpliceAI-don-gain`, `SpliceAI-don-loss` | **SpliceAI**, `07_spliceai.ipynb` |
| `MMSp_acceptor`, `MMSp_donor`, `MMSp_exon`, `MMSp_acceptorIntron` | MMSplice (not covered here) |
| ESM-1v protein language model scores | a sibling of **ESM1b**, `04_esm1b.ipynb` |

The release notes describe those four splice features as ***Masked* SpliceAI** scores —
the same flavour tools/07 builds, for the same reason (Illumina recommends masked for
variant interpretation).

So "CADD agrees with SpliceAI on this splice variant" is close to circular: SpliceAI's
output is one of CADD's inputs. That is not a flaw in either tool, but it does mean the
two are **not** independent evidence, and averaging them or counting them as two votes
overstates the support. The same caution applies to CADD versus ESM1b on missense.

### The version is the anchor, and v1.7 is recent

Elsewhere in this series a release date bounds what could have informed a model. Here the
date is **2024-01-05**, when CADD v1.7 was published — later than every other tool
covered:

| tool | release |
|---|---|
| REVEL | 2016 |
| PrimateAI | 2018 |
| SpliceAI | 2019 |
| EVE | 2021 |
| Pangolin | 2022 |
| AlphaMissense, ESM1b | 2023 |
| **CADD v1.7** | **2024** |

This is worth stating plainly rather than glossing: **CADD v1.7 cannot be anchored to the
2021 CADD-Splice paper.** That paper (Rentzsch et al. 2021, *Genome Medicine*,
PMID 33618777) is **v1.6**; the release notes open by explaining what has changed "Since
the CADD v1.6 release in 2021". v1.7 is Schubach et al. 2024, *Nucleic Acids Research*,
PMID **38183205**, and it added the ESM-1v, regulatory-CNN and Zoonomia conservation
features described above.

Being the most recent tool here means CADD has the **widest window** in which a variant
could have become well-characterised before the model was built — directly through its
annotations, and indirectly through the constituent predictors it inherits. Pin the
version, record it with the score, and treat a CADD-versus-truth-set comparison on
long-known alleles with the same suspicion tools/05 applies to REVEL.

## 4 · Key takeaways

1. **CADD is live, not local.** One API call per variant, no download and no build cell —
   the only tool here with nothing in `data/`. That also makes it the least reproducible:
   cache responses if a run needs to be repeatable.
2. **It is the only score that spans variant classes.** Missense, splice, deep intronic
   and indels all come back on one PHRED scale, which is what makes it a reasonable first
   pass over a mixed CFTR list. **≥ 15 ~ top 3%**, **≥ 20 ~ top 1%**.
3. **One number, no mechanism.** A high CADD says a position looks constrained; it does
   not say *how* the variant does damage. Rank with CADD, then explain with the tool that
   models the mechanism.
4. **Not trained on clinical labels** — proxy-neutral human-derived variants against
   size-matched simulated ones — so low *direct* circularity against ClinVar.
5. **But it contains other tools in this series.** CADD v1.7's features include *masked*
   SpliceAI's four delta scores and ESM-1v protein language model scores, so CADD agreeing
   with SpliceAI (or ESM1b) is partly the same evidence counted twice.
6. **Version is the anchor, and it is 2024.** CADD v1.7 is Schubach et al. 2024
   (PMID 38183205), **not** the 2021 CADD-Splice paper, which is v1.6. It is the most
   recently released tool covered here, so it has the widest window for a variant to have
   been well-described before the model existed.
7. CFTR is **plus-strand**, so coding alleles match the genome directly — no complementing.
   Pin the **genome build** (GRCh38); that is the join key that actually goes wrong.

**This is the last of the tool notebooks.** The cross-tool benchmark over the whole CFTR2
list is written but held back pending the same audit pass.